In [7]:
from collections import defaultdict
from datetime import datetime
import numpy as np
import cv2 as cv
import hashlib
import json
import os

In [8]:
JSON_PATH = "plds.json"
DATASET = "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/organized/PLds"

In [9]:
def get_spot_id(rotated_rect, date):
    flat = [round(x, 4) for item in rotated_rect for x in (item if isinstance(item, list) else [item])]
    combined = flat + date

    combined_string = ','.join(map(str, combined))
    return hashlib.sha256(combined_string.encode()).hexdigest()[:10]


def get_car_id(car_id, date):
    return hashlib.sha256(f"{car_id},{','.join(map(str, date))}".encode()).hexdigest()[:10]


def get_artifacts(ds_path, file):
    with open(f"{ds_path}/{file}", "r") as json_file:
        data = json.load(json_file)

    id_to_date = {img["id"]: img["date"] for img in data["images"]}
    annotations = defaultdict(list)

    for ann in data["annotations"]:
        if ann["car_id"] == -2:
            ann["category_id"] = 0
        
        date = id_to_date[ann["image_id"]]
        ann["spot_id"] = get_spot_id(ann["best_rotated_rect"], date)
        ann["car_id"] = get_car_id(ann["car_id"], date) if ann["category_id"] == 1 else -1
        annotations[ann["image_id"]].append(ann)

    for img in data["images"]:
        img["file_name"] = f"{ds_path}/{img['file_name']}"

    images = sorted(data["images"], key=lambda x: datetime(*x["date"], *x["time"]))
    return images, dict(annotations)

In [10]:
json_data = json.load(open(JSON_PATH, "r"))
all_errors = json_data["errors"]

sorted_errors = dict()
for subset, errors in all_errors.items():
    sorted_errors[subset] = sorted(errors, key=lambda x: (x["gt_duration"] - x["pred_duration"]), reverse=True)[:20]

In [15]:
jsons = [file for file in os.listdir(DATASET) if ".json" in file]

selected_json = jsons[3]
subset = selected_json.replace("_spots.json", "").upper()

os.makedirs(f"_images/{subset}", exist_ok=True)

selected_errors = sorted_errors[subset]
for error in selected_errors:
    error["gt"] = [datetime.strptime(date, "%Y-%m-%d %H:%M:%S") for date in error["gt"]]
    error["pred"] = [datetime.strptime(date, "%Y-%m-%d %H:%M:%S") for date in error["pred"]]

images, annotations = get_artifacts(DATASET, selected_json)
error_dates = [error["gt"][0].date() for error in selected_errors]
error_images = [image for image in images if datetime(*image["date"], *image["time"]).date() in error_dates]

In [16]:
for index, image in enumerate(error_images):
    img_ann = annotations[image["id"]]
    errors_in_day = [error for error in selected_errors if error["spot_id"] in [ann["spot_id"] for ann in img_ann]]
    errors_annotations = [ann for ann in img_ann if ann["spot_id"] in [error["spot_id"] for error in errors_in_day]]

    cv_image = cv.imread(image["file_name"])
    image_date = datetime(*image["date"], *image["time"])
    os.makedirs(f"_images/{subset}/{image_date.date().strftime(format="%Y-%m-%d")}", exist_ok=True)

    for ann in errors_annotations:
        best_rotated_rect = ann["best_rotated_rect"]
        box = np.int64(cv.boxPoints(best_rotated_rect))
        related_error = next((error for error in errors_in_day if error["spot_id"] == ann["spot_id"]), None)
        text_position = tuple(map(int, best_rotated_rect[0]))

        if (related_error["gt"][0] <= image_date <= related_error["gt"][1]) and (related_error["pred"][0] <= image_date <= related_error["pred"][1]):
            cv.drawContours(cv_image, [box], 0, (0, 255, 0), 2)
            cv.putText(cv_image, "GT-PR", text_position, cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv.LINE_AA)
        elif (related_error["gt"][0] <= image_date <= related_error["gt"][1]):
            cv.drawContours(cv_image, [box], 0, (255, 0, 0), 2)
            cv.putText(cv_image, "GT", text_position, cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1, cv.LINE_AA)
        elif (related_error["pred"][0] <= image_date <= related_error["pred"][1]):
            cv.drawContours(cv_image, [box], 0, (0, 255, 255), 2)
            cv.putText(cv_image, "PR", text_position, cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1, cv.LINE_AA)
        else:
            cv.drawContours(cv_image, [box], 0, (128, 128, 128), 2)
            cv.putText(cv_image, "N", text_position, cv.FONT_HERSHEY_SIMPLEX, 0.5, (128, 128, 128), 1, cv.LINE_AA)
    
    cv.imwrite(f"_images/{subset}/{image_date.date().strftime(format="%Y-%m-%d")}/{index+1}.jpg", cv_image)